# 💿 Подключение к Google Drive

Перед началом работы подключите Google Drive и задайте путь к папке **training** на Drive (переменная `PROJECT` в коде; секрет `YSAI_PROJECT` — синхронизируется с репозиторием: внутри должны быть `src/`, `requirements.txt`).

## Метод 1, через Secret:

1. В левой панели перейдите на вкладку «Ключ» (🔑)
2. Добавьте секрет:
    - `YSAI_PROJECT`: `/content/drive/<путь к папке training на Google Drive>`

## Метод 2, напрямую в ячейке с кодом:

`PROJECT = '/content/drive/<путь к папке training на Google Drive>'`

После этого запустите ячейку с кодом.

**Артефакты обучения** (чекпоинты, логи, zip для inference) по умолчанию сохраняются в `{PROJECT}/exports/` — удобно для двусторонней синхронизации Drive ↔ ПК. Свой каталог задаётся в ячейке обучения через `OUTPUT_DIR`.

In [ ]:
# Mount Google Drive and setup paths
PROJECT = ''

if PROJECT == '':
    try:
      from google.colab import userdata
      PROJECT = userdata.get('YSAI_PROJECT')
    except:
      print('Секрет YSAI_PROJECT не установлен')
      raise

if str.isspace(PROJECT) or PROJECT == '':
  raise Exception("Не удалось установить значение переменной PROJECT")

from google.colab import drive
drive.mount('/content/drive')

# Add src directory to Python path
import sys
if f'{PROJECT}/src' not in sys.path:
    sys.path.insert(0, f'{PROJECT}/src')

# Change to project directory
%cd {PROJECT}

# Verify path is added
import os
print(f"✅ Added to sys.path: {PROJECT}/src")
print(f"📁 Current directory: {os.getcwd()}")
print(f"🔍 Available modules in src: {os.listdir('src') if os.path.exists('src') else 'src not found'}")

!pip -q install -U pip
!pip -q install -r requirements.txt

# 🔑 Настройка переменных окружения

Перед запуском блокнота необходимо настроить переменные окружения:

1. Перейдите к иконке ключа (🔑) в левой боковой панели
2. Добавьте следующие секреты:
   - `YSAI_SUPABASE_URL`: URL вашего проекта Supabase
   - `YSAI_SUPABASE_PUBLISHABLE_KEY`: Ваш анонимный ключ Supabase
   - `YSAI_SUPABASE_EMAIL`: Ваш email для входа
   - `YSAI_SUPABASE_PASSWORD`: Ваш пароль для входа
   - `YSAI_DATASET_PATH` (опционально): путь к датасету **без расширения** (к `.lmdb` / `.json` дописывается в ячейке конфигурации)

# ⚙️ Конфигурация

## Установка путей

**Датасет (LMDB)** — в следующей ячейке:
- секрет `YSAI_DATASET_PATH`: базовый путь **без** `.lmdb` / `.json` (например `/content/drive/MyDrive/.../dataset`)
- `LMDB_PATH` / `MAP_PATH`: `''` → из `YSAI_DATASET_PATH` + `.lmdb` / `.json`; непустые значения в ячейке перезаписывают userdata

**Артефакты обучения** — в ячейке «Обучение», переменная `OUTPUT_DIR`:
- `''` (пусто) → `{PROJECT}/exports/` — рекомендуется при синхронизации папки training с ПК
- непустое значение → свой каталог, например `/content/drive/MyDrive/runs`

Внутри базового каталога создаётся `{run_name или timestamp}/` с `checkpoints/`, `logs/`, `plots/` и zip-бандлом модели.

In [ ]:
# Установка путей
# Пусто — из секрета YSAI_DATASET_PATH (без расширения) + .lmdb / .json
LMDB_PATH = ''
MAP_PATH = ''

def _dataset_path_from_userdata():
    try:
        from google.colab import userdata
        base = userdata.get('YSAI_DATASET_PATH')
    except Exception:
        return None
    if not base or not str(base).strip():
        return None
    base = str(base).strip()
    for ext in ('.lmdb', '.json'):
        if base.endswith(ext):
            base = base[: -len(ext)]
    return base

_base = _dataset_path_from_userdata()
if not (LMDB_PATH and str(LMDB_PATH).strip()):
    if _base is None:
        raise Exception(
            'LMDB_PATH не задан: укажите в ячейке или секрет YSAI_DATASET_PATH (путь без расширения)'
        )
    LMDB_PATH = f'{_base}.lmdb'
else:
    LMDB_PATH = str(LMDB_PATH).strip()

if not (MAP_PATH and str(MAP_PATH).strip()):
    if _base is None:
        raise Exception(
            'MAP_PATH не задан: укажите в ячейке или секрет YSAI_DATASET_PATH (путь без расширения)'
        )
    MAP_PATH = f'{_base}.json'
else:
    MAP_PATH = str(MAP_PATH).strip()

print(f'📂 LMDB_PATH: {LMDB_PATH}')
print(f'📂 MAP_PATH: {MAP_PATH}')

# Фильтрация подписей при создании датасета
INPUT_TYPE = 'any'  # Фильтр по типу ввода: 'any', 'mouse', 'touch', или 'pen'

# 🚀 Обучение

Запуск гибридной модели через `TrainingRunner`. Требует готовый LMDB (`LMDB_PATH` из ячейки конфигурации).

## Каталог артефактов

В следующей кодовой ячейке переменная `OUTPUT_DIR`:

- `''` (пусто) → `{PROJECT}/exports/` — рекомендуется при синхронизации папки training с ПК через Drive
- непустое значение → свой каталог, например `/content/drive/MyDrive/runs`

Внутри базового каталога создаётся `{run_name или timestamp}/` с `checkpoints/`, `logs/`, `plots/`; при успешном завершении — zip-бандл для inference.

## Параметры в кодовой ячейке

- `epochs`, `run_name`, `resume` — в `TrainingConfig`
- `FEATURE_PIPELINE` (закомментирован) — список признаков в `DatasetConfig`; по умолчанию из `config.py`

При успешном `runner.run()` zip собирается автоматически; ручная пересборка из checkpoint — в блоке «Сборка model bundle».

In [ ]:
OUTPUT_DIR = ''

import os

if not (OUTPUT_DIR and str(OUTPUT_DIR).strip()):
    OUTPUT_DIR = os.path.join(PROJECT, 'exports')
else:
    OUTPUT_DIR = str(OUTPUT_DIR).strip()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'📦 Artifacts base: {OUTPUT_DIR}')

# FEATURE_PIPELINE = [
#     "x", "y", "p", "t",
#     "vx", "vy", "ax", "ay", "prate", "path_tangent_angle", "abs_delta_pressure"
# ]

from config import DatasetConfig, ModelConfig, TrainingConfig
from training import TrainingRunner

# Dataset configuration
ds_cfg = DatasetConfig(
    lmdb_path = LMDB_PATH,
    # feature_pipeline = FEATURE_PIPELINE,
    dataset_sample_ratio = 1
)

# Model configuration
model_cfg = ModelConfig(

)

# Training configuration
train_cfg = TrainingConfig(
  output_dir = OUTPUT_DIR,
  epochs = 150,
  run_name = None,
  resume = True
)

print("🚀 Starting Hybrid Model Training\n")

try:
    runner = TrainingRunner(
        dataset_cfg=ds_cfg, model_cfg=model_cfg, train_cfg=train_cfg
    )
    runner.run()

    print("\n" + "=" * 80)
    print("✅ Training completed successfully!")
    print("=" * 80)

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")

except Exception as e:
    print("\n" + "=" * 80)
    print("❌ Training failed with error:")
    print("=" * 80)
    print(f"\n{type(e).__name__}: {e}\n")
    import traceback

    traceback.print_exc()
    exit(1)

# 📦 Сборка model bundle из готового run

Запускайте после обучения (в том числе если цикл упал, но остались `checkpoints/best_by_eer.pt` и `model.py`).

- `RUN_DIR` — путь к папке run (`.../exports/sig-v3/`). Пусто → **последний** экспортируемый run.
- `BUNDLE_NAME` — имя zip (`sig-v3.zip`). Пусто → имя папки run.
- База артефактов: из ячейки «Обучение» (`OUTPUT_DIR`), иначе `{PROJECT}/exports`, иначе можно задать `OUTPUT_DIR` в кодовой ячейке ниже.

При успешном `runner.run()` zip уже создаётся автоматически; эта ячейка — для ручной пересборки или после сбоя.

In [ ]:
# 📦 Export model bundle from existing run
RUN_DIR = ''       # e.g. '/content/drive/.../exports/sig-v3' or leave empty for latest
BUNDLE_NAME = ''   # e.g. 'sig-v3' or leave empty → folder name
# Если ячейку «Обучение» не запускали — задайте базу артефактов явно:
# OUTPUT_DIR = '/content/drive/MyDrive/.../exports'

import os
from pathlib import Path

from training.export_bundle import export_bundle_from_run, resolve_latest_run_dir

try:
    _artifacts_base = OUTPUT_DIR
except NameError:
    _artifacts_base = ''

if not (_artifacts_base and str(_artifacts_base).strip()):
    try:
        _project = PROJECT
    except NameError:
        _project = os.getcwd()
    _artifacts_base = os.path.join(_project, 'exports')

OUTPUT_DIR = str(_artifacts_base).strip()
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'📦 Artifacts base: {OUTPUT_DIR}')

if RUN_DIR and str(RUN_DIR).strip():
    run_path = Path(RUN_DIR).expanduser().resolve()
else:
    latest = resolve_latest_run_dir(OUTPUT_DIR)
    if latest is None:
        raise FileNotFoundError(
            f'No exportable run under {OUTPUT_DIR} '
            '(need checkpoints/best_by_eer.pt and model.py)'
        )
    run_path = latest
    print(f'📂 Latest run: {run_path}')

bundle_name = (BUNDLE_NAME or run_path.name).strip()
zip_path = export_bundle_from_run(str(run_path), bundle_name=bundle_name or None)
print(f'✅ Bundle written: {zip_path}')

In [ ]:

# Проверка готовности

# Проверка работы импортов
try:
    import analysis
    import data
    import models
    import training
    import utils
    import config
    print("✅ Все модули успешно импортированы!")
except ImportError as e:
    print(f"❌ Ошибка импорта: {e}")
    print("Доступные пути:", sys.path)

# Проверка переменных окружения
from google.colab import userdata
print(f"\n🔑 YSAI_SUPABASE_URL: {'✅ Установлена' if userdata.get('YSAI_SUPABASE_URL') else '❌ Отсутствует'}")
print(f"🔑 YSAI_SUPABASE_PUBLISHABLE_KEY: {'✅ Установлена' if userdata.get('YSAI_SUPABASE_PUBLISHABLE_KEY') else '❌ Отсутствует'}")
print(f"🔑 YSAI_SUPABASE_EMAIL: {'✅ Установлена' if userdata.get('YSAI_SUPABASE_EMAIL') else '❌ Отсутствует'}")
print(f"🔑 YSAI_SUPABASE_PASSWORD: {'✅ Установлена' if userdata.get('YSAI_SUPABASE_PASSWORD') else '❌ Отсутствует'}")
print(f"🔑 YSAI_DATASET_PATH: {'✅ Установлен' if userdata.get('YSAI_DATASET_PATH') else '❌ Отсутствует'}")


# Вывод конифигураций
from os import path
print(f"\nLMDB_PATH: {LMDB_PATH} ({"существует" if path.exists(LMDB_PATH) else "не существует"})")
print(f"MAP_PATH: {MAP_PATH} ({"существует" if path.exists(MAP_PATH) else "не существует"})")
print(f"INPUT_TYPE: {INPUT_TYPE}")


In [ ]:
# Создание датасета

from data import build_lmdb_from_supabase
from google.colab import userdata
build_lmdb_from_supabase(
    supabase_url=userdata.get('YSAI_SUPABASE_URL'),
    anon_key=userdata.get('YSAI_SUPABASE_PUBLISHABLE_KEY'),
    email=userdata.get('YSAI_SUPABASE_EMAIL'),
    password=userdata.get('YSAI_SUPABASE_PASSWORD'),
    output_lmdb_path=LMDB_PATH,
    output_map_json_path=MAP_PATH,
    input_type=INPUT_TYPE,
)
print('LMDB записан в', LMDB_PATH)
print('MAP записан в', MAP_PATH)


In [ ]:
# Отчет по Supabase
from analysis import analyze_data, DataSource, print_supabase_report
print("Анализ...")
supabase_results = analyze_data(DataSource.SUPABASE,
    supabase_url=userdata.get('YSAI_SUPABASE_URL'),
    anon_key=userdata.get('YSAI_SUPABASE_PUBLISHABLE_KEY'),
    email=userdata.get('YSAI_SUPABASE_EMAIL'),
    password=userdata.get('YSAI_SUPABASE_PASSWORD')
)
print("=== ОТЧЕТ ПО SUPABASE ===")
print_supabase_report(supabase_results)


In [ ]:
# Отчет по LMDB
# Перезагружаем модуль для избежания проблем с кэшированием
import importlib
import analysis
importlib.reload(analysis)

from analysis import analyze_data, DataSource, print_lmdb_report
print("Анализ LMDB...")
lmdb_results = analyze_data(DataSource.LMDB, lmdb_path=LMDB_PATH)
print("=== ОТЧЕТ ПО LMDB ===")
print_lmdb_report(lmdb_results)

In [ ]:
# 📊 Визуализация распределения t для LMDB
from analysis import plot_t_distribution_by_label

fig, text = plot_t_distribution_by_label(LMDB_PATH, bin_width=500, text_step=100, overlay=True)
fig.show()
print(text)

In [ ]:
# 🔍 Поиск неподрезанных подписей с мусорными точками
from analysis import find_untrimmed_signatures

problematic = find_untrimmed_signatures(LMDB_PATH, max_skip_check=10)
print(f"Найдено {len(problematic)} подозрительных подписей")
# Выведем первые 10 результатов
for rec in problematic[:10]:
    print(rec)


In [ ]:
from google.colab import runtime

runtime.unassign()